Applay Feature Extraction

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files

files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

In [ ]:
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/skin_data

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/skin_data/HAM10000_metadata.csv"
)

print(df.head())
print(df["dx"].value_counts())

In [ ]:
print(df.columns)

In [ ]:
df = df[df["dx"] != "akiec"].copy()

df["label"] = df["dx"].apply(
    lambda x: 1 if x in ["mel", "bcc"] else 0
)

print(df["label"].value_counts())

In [ ]:
import keras
from keras.applications.resnet50 import ResNet50, preprocess_input
from keras.utils import load_img, img_to_array
import numpy as np

# Load model
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

In [ ]:
!find /content/skin_data -name "*.jpg" | head

In [ ]:
image_path ="/content/skin_data/ham10000_images_part_2/ISIC_0032637.jpg"

img = load_img(image_path, target_size=(224, 224))

img = img_to_array(img)

img = np.expand_dims(img, axis=0)

img = preprocess_input(img)

features = base_model.predict(img, verbose=0)

print(features.shape)

In [ ]:
from pathlib import Path
import numpy as np

image_dirs = [
    Path("/content/skin_data/ham10000_images_part_1"),
    Path("/content/skin_data/ham10000_images_part_2")
]

# Create a dictionary mapping image IDs to paths
image_paths = {
    img.stem: img
    for folder in image_dirs
    for img in folder.glob("*.jpg")
}

X_features = []
y_labels = []
batch_size = 32

for start in range(0, len(df), batch_size):

    batch_df = df.iloc[start:start + batch_size]
    batch_images = []

    for _, row in batch_df.iterrows():

        image_id = row["image_id"]
        image_path = image_paths.get(image_id)

        if image_path is None:
            raise FileNotFoundError(f"Image not found: {image_id}")

        img = load_img(image_path, target_size=(224, 224))
        img = img_to_array(img)
        batch_images.append(img)

    batch_images = np.array(batch_images)
    batch_images = preprocess_input(batch_images)

    batch_features = base_model.predict(
        batch_images,
        verbose=0
    )

    X_features.append(batch_features)
    y_labels.extend(batch_df["label"].values)

X_features = np.concatenate(X_features, axis=0)
y_labels = np.array(y_labels)

print("Features:", X_features.shape)
print("Labels:", y_labels.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

classifier = make_pipeline(
    StandardScaler(),
    SVC(class_weight="balanced")
)

classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "Malignant"]
))

In [ ]:
import joblib

joblib.dump(classifier, "/content/skin_svm.pkl")

In [ ]:
base_model.save("/content/resnet50_feature_extractor.keras")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from keras.applications.resnet50 import preprocess_input
from keras.utils import load_img, img_to_array
from keras.models import load_model
import numpy as np

# Load the same feature extractor
base_model = load_model("/content/resnet50_feature_extractor.keras")

# Load image
img = load_img(uploaded, target_size=(224, 224))
img = img_to_array(img)

# Add batch dimension
img = np.expand_dims(img, axis=0)

# Preprocessing
img = preprocess_input(img)

# Extract features
features = base_model.predict(img, verbose=0)

print(features.shape)

In [ ]:
import joblib

classifier = joblib.load("/content/skin_svm.pkl")

prediction = classifier.predict(features)

if prediction == 0:
    print("Predicted class: Benign")
else:
    print("Predicted class: Malignant")